# BRD4 BD1 + (+)-JQ1: a first molecular dynamics run with OpenMM

A beginner-friendly notebook. Run it cell by cell; each step explains **what** it does and **what to look at**.

## The system
- **BRD4 BD1**: a protein domain that binds acetylated lysines on histones. It is a cancer drug target.
- **(+)-JQ1**: a small inhibitor that sits in the acetyl-lysine pocket. Its key contact is a **hydrogen bond with Asn140**.
- **PDB 3MXF**: the crystal structure of the complex.

## What is a molecular dynamics (MD) simulation?
1. Start from a static structure (the crystal).
2. Give it a **force field** (the "physics": bonds, angles, charges, van der Waals).
3. Put it in salty water at 300 K.
4. Move the atoms forward in 2 fs steps and save the "movie" (trajectory).
5. Analyze the movie: does the ligand stay in the pocket? Does it keep the Asn140 contact?

## Pipeline
| Part | What it does | Output |
|---|---|---|
| 1. Preparation | Clean the protein, set up the ligand, add water | `system.pdb`, `system.xml` |
| 2. Simulation | Minimize, heat, equilibrate, run (3 replicas) | `traj_seed*.dcd` |
| 3. Analysis | RMSD, RMSF, contacts, PCA | `figures/` |

## 0. Check the environment

You need the environment from `environment.yml` (see the README for Windows). This cell imports everything and lists the OpenMM platforms: **CUDA** is the fast one (GPU); **CPU** works but is very slow for 20 ns.

In [ ]:
import glob, urllib.request
from io import StringIO
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
from MDAnalysis.analysis import align, distances, rms
from openff.toolkit import Molecule
from openff.toolkit.utils.nagl_wrapper import NAGLToolkitWrapper
from openmm import CustomExternalForce, LangevinMiddleIntegrator, MonteCarloBarostat, XmlSerializer, unit
from openmm import Platform
from openmm.app import DCDFile, HBonds, Modeller, PDBFile, PME, Simulation
from openmmforcefields.generators import SystemGenerator
from pdbfixer import PDBFixer
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

print([Platform.getPlatform(i).getName() for i in range(Platform.getNumPlatforms())])
Path("figures").mkdir(exist_ok=True)

# Part 1: Prepare the system

A crystal PDB file **cannot be simulated as is**: it lacks hydrogens and some atoms, contains crystal waters and additives, and the ligand has no bond orders or charges. We fix all of that.

## 1.1 Download the structure

We download 3MXF from RCSB. The ligand `HETATM` lines (residue `JQ1`, chain A) are kept separately, because PDBFixer does not know what JQ1 is.

In [ ]:
PDB_ID, LIG = "3MXF", "JQ1"
JQ1_SMILES = "CC1=C(SC2=C1C(=N[C@H](C3=NN=C(N32)C)CC(=O)OC(C)(C)C)C4=CC=C(C=C4)Cl)C"

raw = urllib.request.urlopen(f"https://files.rcsb.org/download/{PDB_ID}.pdb").read().decode()
lig_lines = [l for l in raw.splitlines() if l.startswith("HETATM") and l[17:20] == LIG and l[21] == "A"]
assert lig_lines, f"{LIG} not found in chain A of {PDB_ID}; check rcsb.org"
print(len(lig_lines), "JQ1 heavy atoms in chain A")

## 1.2 Fix the protein (PDBFixer)

Steps, in order:
- **Chain A only**: the crystal has several copies; one is enough.
- **Remove heterogens and waters**: drops crystal waters, ions and additives (including JQ1, which we add back later).
- **Missing residues**: disordered ends are not modelled (`missingResidues = {}`), a deliberate simplification.
- **Missing atoms + hydrogens at pH 7.4**: X-rays barely see hydrogens, so PDBFixer adds them for pH 7.4.

In [ ]:
fixer = PDBFixer(pdbfile=StringIO(raw))
fixer.removeChains([c.index for c in fixer.topology.chains() if c.id != "A"])
fixer.removeHeterogens(keepWater=False)
fixer.findMissingResidues()
fixer.missingResidues = {}  # disordered ends are not modelled
fixer.findNonstandardResidues(); fixer.replaceNonstandardResidues()
fixer.findMissingAtoms(); fixer.addMissingAtoms()
fixer.addMissingHydrogens(7.4)
print("protein atoms:", fixer.topology.getNumAtoms())

## 1.3 The ligand: from coordinates to a chemical molecule

The PDB only has atom positions, **no bond orders** (double or single?) and no hydrogens. To set up the force field we need the real chemistry, so:

1. Read the coordinates with RDKit.
2. **`AssignBondOrdersFromTemplate`**: copy the bond orders from the SMILES string (the chemical "template") onto the crystal coordinates.
3. `AddHs(addCoords=True)`: add hydrogens in sensible positions.
4. Check the **stereochemistry**: the SMILES has `[C@H]`, which is (+)-JQ1 (the active form; (−)-JQ1 is inactive). If it does not match, the `assert` fails.
5. Partial charges with **NAGL**, a neural network that mimics AM1-BCC charges (no AmberTools needed, which is not available on Windows).

In [ ]:
rd = Chem.MolFromPDBBlock("\n".join(lig_lines), removeHs=False)
rd = AllChem.AssignBondOrdersFromTemplate(Chem.MolFromSmiles(JQ1_SMILES), rd)
rd = Chem.AddHs(rd, addCoords=True)
Chem.AssignStereochemistryFrom3D(rd)
for at in rd.GetAtoms():
    at.SetMonomerInfo(None)  # otherwise the added H end up in separate residues
lig = Molecule.from_rdkit(rd)
assert lig.to_smiles() == Molecule.from_smiles(JQ1_SMILES).to_smiles(), "ligand stereo mismatch: not (+)-JQ1"
lig.assign_partial_charges("openff-gnn-am1bcc-0.1.0-rc.1.pt", toolkit_registry=NAGLToolkitWrapper())
print("ligand atoms:", lig.n_atoms, "| total charge:", lig.total_charge)

## 1.4 Force field and solvation

**Force field** (the physics of the model):
- Protein: **Amber14** (`amber14-all.xml`).
- Water: **TIP3P**.
- Ligand: **OpenFF Sage 2.1** (`openff-2.1.0`), with the NAGL charges from above.

`SystemGenerator` combines the three into one system.

**Box** (`Modeller.addSolvent`):
- 1 nm of water around the solute.
- 0.15 M NaCl (physiological) and charge neutralization.

**Simulation settings**:
- **PME**: handles long-range electrostatics (1 nm cutoff for the rest).
- **HBonds + rigidWater**: bonds to hydrogen and the water geometry are held fixed, which allows a **2 fs** time step.

In [ ]:
gen = SystemGenerator(
    forcefields=["amber14-all.xml", "amber14/tip3p.xml"],
    small_molecule_forcefield="openff-2.1.0",
    molecules=[lig],
    forcefield_kwargs=dict(constraints=HBonds, rigidWater=True, removeCMMotion=True),
    periodic_forcefield_kwargs=dict(nonbondedMethod=PME, nonbondedCutoff=1 * unit.nanometer),
)

m = Modeller(fixer.topology, fixer.positions)
m.add(lig.to_topology().to_openmm(), lig.conformers[0].to_openmm())
m.addSolvent(gen.forcefield, model="tip3p", padding=1 * unit.nanometer,
             ionicStrength=0.15 * unit.molar, neutralize=True)

system = gen.create_system(m.topology, molecules=[lig])
PDBFile.writeFile(m.topology, m.positions, open("system.pdb", "w"), keepIds=True)
open("system.xml", "w").write(XmlSerializer.serialize(system))
print("total atoms:", m.topology.getNumAtoms(),
      "| residues (no water):", sorted({r.name for r in m.topology.residues()} - {"HOH"})[-5:])

**What to look at**: the total atom count (tens of thousands, mostly water) and that `NA`, `CL` and the ligand residue (`UNK`) appear among the residues. We save `system.pdb` (topology + coordinates) and `system.xml` (all force field parameters) so Part 2 does not repeat any of this.

*Optional*: `import nglview as nv; nv.show_file("system.pdb")` to view it in 3D.

# Part 2: Simulation

## Phases (standard equilibration protocol)

Jumping straight to 300 K from the raw structure would blow the system up. We go step by step:

| Phase | What it does | Why |
|---|---|---|
| Minimization | Lowers the energy by removing atomic clashes | The prepared structure has overlaps |
| NVT 100 ps | Heats from 50 to 300 K in 10 steps, fixed volume | Raise the temperature gently |
| NPT 1 ns | Adds a barostat (1 bar); restraints go 1000→100→10→0 | The box finds its density and the protein relaxes slowly |
| Production 20 ns | No restraints, one frame saved every 10 ps | The data we will analyze |

**Position restraints**: a spring tying each protein heavy atom to its starting position, with force constant `k` (kJ/mol/nm²). Strong at first (1000) so the water settles around the protein; zero at the end. Not applied to the ligand (with k=1000 its hydrogens blew up).

**Replicas**: we use several seeds (different random starting velocities). If 3 independent replicas tell the same story, the result is more believable than with one.

## Parameters
`SCALE` multiplies **all** durations. Start with `0.05` (quick test, ~1 min on a GPU) and set `SCALE = 1.0` once everything works.

In [ ]:
SEEDS = [1, 2, 3]
NS = 20          # ns of production per replica
SCALE = 0.05     # 0.05 = quick test; 1.0 = full run (~2 h per seed on an RTX 2060 Super)
DT, T = 0.002, 300  # ps, K
steps = lambda ps: max(1, int(ps * SCALE / DT))

## Function `run(seed)`

Reads the saved system and runs the four phases. Things to notice:

- `CustomExternalForce` defines the restraint spring. `k` is a **global parameter**, so changing it between stages is just a `setParameter`.
- `LangevinMiddleIntegrator` is the thermostat: it keeps the temperature on target. During heating we raise the target temperature every 10 ps.
- The `MonteCarloBarostat` is added **after** the NVT stage; `reinitialize(preserveState=True)` makes the context pick it up without losing positions or velocities.
- In production we save **only the solute** (no water or ions) to a DCD file: much smaller files, and enough for the analysis.

In [ ]:
def run(seed):
    pdb = PDBFile("system.pdb")
    system = XmlSerializer.deserialize(open("system.xml").read())

    # Position restraints on protein heavy atoms (not the UNK ligand)
    rest = CustomExternalForce("0.5*k*((x-x0)^2+(y-y0)^2+(z-z0)^2)")
    rest.addGlobalParameter("k", 1000.0)  # kJ/mol/nm^2
    for p in ("x0", "y0", "z0"):
        rest.addPerParticleParameter(p)
    solute = [r for r in pdb.topology.residues() if r.name not in ("HOH", "NA", "CL")]
    for r in solute:
        for at in r.atoms():
            if at.element.symbol != "H" and r.name != "UNK":
                rest.addParticle(at.index, pdb.positions[at.index].value_in_unit(unit.nanometer))
    system.addForce(rest)

    integ = LangevinMiddleIntegrator(50 * unit.kelvin, 1 / unit.picosecond, DT * unit.picosecond)
    integ.setRandomNumberSeed(seed)
    sim = Simulation(pdb.topology, system, integ)  # OpenMM picks the fastest platform (CUDA if available)
    print("platform:", sim.context.getPlatform().getName())
    sim.context.setPositions(pdb.positions)

    sim.minimizeEnergy()
    sim.context.setVelocitiesToTemperature(50 * unit.kelvin, seed)

    # NVT: heat 50 -> 300 K in 10 chunks of 10 ps, k = 1000
    for i in range(1, 11):
        integ.setTemperature((50 + 25 * i) * unit.kelvin)
        sim.step(steps(10))

    # NPT: add barostat and release restraints in stages (250 ps each)
    baro = MonteCarloBarostat(1 * unit.bar, T * unit.kelvin, 25)
    baro.setRandomNumberSeed(seed)
    system.addForce(baro)
    sim.context.reinitialize(preserveState=True)
    for k in (1000, 100, 10, 0):
        sim.context.setParameter("k", k)
        sim.step(steps(250))

    # Production: solute only, one frame every 10 ps
    keep = Modeller(pdb.topology, pdb.positions)
    keep.delete([r for r in pdb.topology.residues() if r.name in ("HOH", "NA", "CL")])
    idx = [at.index for r in solute for at in r.atoms()]
    PDBFile.writeFile(keep.topology, keep.positions, open("solute.pdb", "w"), keepIds=True)
    every = steps(10)
    n_frames = int(NS * 1000 / 10 * SCALE)
    with open(f"traj_seed{seed}.dcd", "wb") as f:
        dcd = DCDFile(f, keep.topology, 10 * unit.picosecond)
        for _ in range(n_frames):
            sim.step(every)
            pos = sim.context.getState(getPositions=True).getPositions(asNumpy=True)
            dcd.writeModel(pos[idx])
    # no checkpoints; if it fails, rerun that seed

## Run the replicas

Each call blocks the notebook until it finishes. With `SCALE = 1.0` it takes hours per seed: run one seed per cell, or launch `python 02_run.py --seed N` in a terminal (the original script is still in the folder) and jump to Part 3.

In [ ]:
for s in SEEDS:
    run(s)

# Part 3: Analysis

We load each trajectory and compute:

| Metric | Question it answers |
|---|---|
| **RMSD** of CA atoms (protein) | Does it stay folded? Stable = a curve that flattens out |
| **Ligand RMSD** (after aligning the protein) | Does it stay in the crystal pose, or move/leave? |
| **RMSF** per residue | Which regions are flexible (ZA/BC loops) and which are rigid? |
| **Asn140 ND2 to ligand N/O distance** | Is the key H-bond kept? (< 3.5 Å = yes) |
| **Contact occupancy** | Which residues touch the ligand, and for what fraction of the time? |
| **PCA + KMeans** | Which large motions dominate? Do the replicas visit the same states? |

**Details**:
- Everything is aligned on the CA atoms to the first frame of the first replica, to remove overall translation and rotation.
- The H-bond is a **distance-only proxy** (no angle): a simple approximation.
- RMSD is in Å; time is 0.01 ns per frame.

In [ ]:
SEEDS = sorted(int(f[9:-4]) for f in glob.glob("traj_seed*.dcd"))
LIGSEL = "not protein"  # solute.pdb only has protein + ligand
N140_ND2 = "resid 140 and name ND2"
POCKET_CUTOFF = 4.0   # Å: residue closer than this to the ligand = contact
HBOND_CUTOFF = 3.5    # Å: H-bond proxy between heavy atoms N/O

ref = mda.Universe("solute.pdb", f"traj_seed{SEEDS[0]}.dcd")
ref.trajectory[0]
assert ref.select_atoms(N140_ND2).n_atoms == 1, "Asn140 ND2 not found: check residue numbering"
assert ref.select_atoms(LIGSEL).n_atoms > 0, "no ligand found"
print("replicas found:", SEEDS)

### 3.1 Compute the metrics

For each seed:
1. `rms.RMSD` with `groupselections` gives the CA RMSD (column 2) and the heavy-atom ligand RMSD (column 3) in one go, fitting on the CA atoms.
2. `AlignTraj` aligns the whole trajectory (in memory) so RMSF and PCA are consistent.
3. Minimum Asn140 ND2 to ligand N/O distance in each frame.
4. Occupancy: in each frame, which residues are within 4 Å of the ligand; averaged over frames.

In [ ]:
rmsd, rmsf, nh, occ, ca_xyz = {}, {}, {}, {}, {}
for s in SEEDS:
    u = mda.Universe("solute.pdb", f"traj_seed{s}.dcd")
    t_ns = np.arange(len(u.trajectory)) * 0.01
    R = rms.RMSD(u, ref, select="name CA", groupselections=[f"({LIGSEL}) and not name H*"]).run()
    rmsd[s] = (t_ns, R.results.rmsd[:, 2], R.results.rmsd[:, 3])  # CA, ligand

    align.AlignTraj(u, ref, select="name CA", in_memory=True).run()
    ca = u.select_atoms("name CA")
    rmsf[s] = (ca.resids, rms.RMSF(ca).run().results.rmsf)
    ca_xyz[s] = np.array([ca.positions.ravel() for _ in u.trajectory])

    lig, prot = u.select_atoms(f"({LIGSEL}) and not name H*"), u.select_atoms("protein")
    n140, no = u.select_atoms(N140_ND2), lig.select_atoms("element N O")
    nh[s] = np.array([distances.distance_array(n140.positions, no.positions).min() for _ in u.trajectory])
    resids = np.unique(prot.resids)
    contacts = np.zeros(len(resids))
    for _ in u.trajectory:
        near = prot.select_atoms(f"around {POCKET_CUTOFF} group lig", lig=lig).resids
        contacts += np.isin(resids, near)
    occ[s] = (resids, contacts / len(u.trajectory))
    print(f"seed {s}: ligand RMSD {rmsd[s][2].mean():.2f} A | Asn140 H-bond occupancy {(nh[s] < HBOND_CUTOFF).mean():.2f}")

### 3.2 RMSD and RMSF

**How to read it**:
- *CA RMSD*: typically 1–2 Å for a stable domain. A steady rise that never flattens suggests it has not equilibrated.
- *Ligand RMSD*: low values (~1–2 Å) = still in the crystal pose. Jumps = it changed pose.
- *RMSF*: peaks at the ends and in loops; minima in the core helices.
- If the replicas differ a lot, there is little sampling or real state changes.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for s in SEEDS:
    ax[0].plot(rmsd[s][0], rmsd[s][1], label=f"seed {s}")
    ax[1].plot(rmsd[s][0], rmsd[s][2])
    ax[2].plot(*rmsf[s])
for a_, t in zip(ax, ["Protein CA RMSD (A)", "Ligand RMSD, protein-aligned (A)", "CA RMSF (A)"]):
    a_.set_title(t)
ax[0].set_xlabel("ns"); ax[1].set_xlabel("ns"); ax[2].set_xlabel("residue"); ax[0].legend()
fig.tight_layout(); fig.savefig("figures/rmsd_rmsf.png", dpi=150)

### 3.3 Asn140 contact and pocket

**How to read it**:
- *Left*: minimum Asn140 ND2 to ligand N/O distance; below the dashed line (3.5 Å) there is an (approximate) H-bond.
- *Right*: fraction of time each residue is within 4 Å of the ligand. We expect peaks in the acetyl-lysine pocket (Trp81, Pro82, Phe83, Leu92, Leu94, Tyr97, Cys136, Asn140, Ile146...).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for s in SEEDS:
    ax[0].plot(rmsd[s][0], nh[s], label=f"seed {s}")
    ax[1].plot(*occ[s])
ax[0].axhline(HBOND_CUTOFF, ls="--", c="k"); ax[0].set_title("Asn140 ND2 - ligand N/O min distance (A)"); ax[0].set_xlabel("ns"); ax[0].legend()
ax[1].set_title(f"Pocket contact occupancy (<{POCKET_CUTOFF} A)"); ax[1].set_xlabel("residue")
fig.tight_layout(); fig.savefig("figures/contacts.png", dpi=150)

### 3.4 PCA and clustering

- **PCA**: reduces the 3·N_CA coordinates of each frame to a few "modes" that hold most of the variance. PC1 = the largest overall motion. We pool the replicas to compare them in the same space.
- **KMeans (k=3)**: groups the frames into 3 rough conformations in the space of the first 3 PCs.
- **What to look at**: if all replicas overlap in the cloud, they explore the same thing. If each sits in a different area, there is no convergence. Cluster populations are **descriptive only** (20 ns × 3 is short).

In [ ]:
X = np.vstack([ca_xyz[s] for s in SEEDS])
lab = np.concatenate([[s] * len(ca_xyz[s]) for s in SEEDS])
pcs = PCA(n_components=3).fit_transform(X)
cl = KMeans(n_clusters=min(3, len(X)), n_init=10, random_state=0).fit_predict(pcs)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for s in SEEDS:
    ax[0].scatter(*pcs[lab == s, :2].T, s=4, label=f"seed {s}")
ax[1].scatter(*pcs[:, :2].T, s=4, c=cl)
for a_, t in zip(ax, ["PCA by replica", "KMeans clusters (k=3)"]):
    a_.set_title(t); a_.set_xlabel("PC1"); a_.set_ylabel("PC2")
ax[0].legend(); fig.tight_layout(); fig.savefig("figures/pca_clusters.png", dpi=150)
print("cluster occupancy per replica:")
for s in SEEDS:
    print(f"  seed {s}:", np.bincount(cl[lab == s], minlength=3) / (lab == s).sum())

# Limitations

This is a learning project, so it is kept simple:
- 20 ns × 3 replicas is short; no conclusions about convergence or free energies.
- Only one force field combination (Amber14 + OpenFF Sage).
- Disordered ends are not modelled, and crystal waters are removed.
- The H-bond is checked by distance only (no angle).
- No apo control and no checkpoints.
